# Gemma 4 31B V7.5: All 60 Layers Spatial Semantic Districts + GTA LOD Residual Split (A100-SXM4-80GB Max Util)

**Method 1 Full-Model Conversion (`v5-qat-dev`):**
- **Zero 1-Bit Attention (`L00..L59`):** 5-trit / 3-bit Attention everywhere.
- **All 56 Cold Middle MLPs (`L02..L57`):** Co-activation Spatial Districts (`Zone 0 Shared Core = 12 blocks = 3,072 neurons` + `12` semantic city districts `= 72 blocks = 18,432 neurons`, `1,536` neurons per district), optional real-backprop District-Sparsity Distillation, Prompt-Level District Router, and Rotated-Basis **GTA LOD Residual Split** (`W_base 1b` Row-Major pinned + `ΔW_hi-res 2b` across `lod_shared.t16c` & `lod_d0..lod_d11.t16c`).
- **Pi 4 Envelope (`<= 7.10 GiB <= 7.6 GiB`):** Base LM (`~5.14 GiB`, Vision `223 MiB` evicted via `MADV_DONTNEED`) + Pinned Shared Core (`0.67 GiB`) + Top-2 Active District Arena (`0.67 GiB`) + `GlyphKV-LOD` + `Assistant-V7 Q2` + `CRTR`/`C8DB`, backed by **`~10.02 GiB`** of 1b+2b weights.
- **Maxed A100-80GB Execution:** Full HBM2e activation residency (`V7_5_RESIDENT_GPU=1`), TF32 Tensor Cores, `micro_batch=128`, `als_ev_toks=4096`, async `safetensors` layer prefetch, and zero-hole `.partial`-first Drive checkpointing.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/DannyB-bit/Language-U-Gemma4.git"
REPO_DIR = Path("/content/Language-U-Gemma4")
DRIVE_V75 = Path("/content/drive/MyDrive/gemma4_v7_5_spatial_districts")
DRIVE_V75.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-b", "v5-qat-dev", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "v5-qat-dev"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/v5-qat-dev"], check=True)

print(subprocess.check_output(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], text=True).strip(), flush=True)
print(subprocess.check_output(["nvidia-smi"], text=True), flush=True)

import torch
assert torch.cuda.is_available(), "CUDA GPU required! Switch Runtime -> Change runtime type -> A100 High-RAM."
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
smoke_t = (torch.ones((16, 16), device="cuda", dtype=torch.bfloat16) @ torch.ones((16, 16), device="cuda", dtype=torch.bfloat16)).sum().item()
print(f"[GPU Ready] {torch.cuda.get_device_name(0)} | VRAM={vram_gb:.2f} GiB | max_mode={vram_gb >= 64.0} | bf16_smoke={smoke_t:.1f}", flush=True)

_need_pip = False
try:
    import datasets, safetensors, tokenizers, huggingface_hub, PIL
    from transformers.models.gemma4.modeling_gemma4 import Gemma4VisionModel
except Exception:
    _need_pip = True
if _need_pip:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "transformers>=4.51.0", "accelerate", "safetensors", "huggingface_hub", "datasets", "tokenizers", "pillow"],
        check=True,
    )

try:
    from google.colab import userdata
    hf_tok = userdata.get("HF_TOKEN")
    if hf_tok:
        os.environ["HF_TOKEN"] = hf_tok
        print("Loaded HF_TOKEN from Colab secrets.")
except Exception:
    pass

SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v7_5_spatial_districts_a100.py"
print("Running 6-gate V7.5 Spatial Districts CPU pre-flight selftest...", flush=True)
print(subprocess.check_output([sys.executable, str(SCRIPT), "--selftest"], text=True), flush=True)

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_DIR = Path("/content/Language-U-Gemma4")
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "v5-qat-dev"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/v5-qat-dev"], check=True)
SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v7_5_spatial_districts_a100.py"
LOG_PATH = Path("/content/v7_5_train.log")
DRIVE_V75 = Path("/content/drive/MyDrive/gemma4_v7_5_spatial_districts")

import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
env = os.environ.copy()
env["V7_5_DRIVE_DIR"] = str(DRIVE_V75)
env["V7_5_OUT_DIR"] = "/content/v7_5_outputs"
env["V7_5_A100_MAX_UTIL"] = "1"
env["V7_5_RESIDENT_GPU"] = "1" if vram_gb >= 64.0 else "0"
env["V7_5_DISTRICT_DISTILL"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"
env["COLAB_AUTO_UNASSIGN"] = "1"
env["PYTHONUNBUFFERED"] = "1"

print(f"Launching ALL-60-LAYER V7.5 Spatial Districts + GTA LOD (Max VRAM {vram_gb:.1f} GiB) -> {LOG_PATH}...", flush=True)
with open(LOG_PATH, "w", encoding="utf-8") as lfp:
    proc = subprocess.Popen([sys.executable, "-u", str(SCRIPT), "--run"], stdout=lfp, stderr=subprocess.STDOUT, env=env)

seen = 0
while proc.poll() is None:
    time.sleep(15)
    if LOG_PATH.is_file():
        txt = LOG_PATH.read_text(encoding="utf-8", errors="replace")
        if len(txt) > seen:
            print(txt[seen:], end="", flush=True)
            seen = len(txt)

txt = LOG_PATH.read_text(encoding="utf-8", errors="replace")
if len(txt) > seen:
    print(txt[seen:], end="", flush=True)
assert proc.returncode == 0, f"V7.5 pipeline exited with code {proc.returncode}"

sys.path.insert(0, "/content/Language-U-Gemma4/experiments/run270_rust/calib/31b")
import run_v7_0_fullrank_crtr_gpu as v70
import train_cortical_router as crtr

rpt = json.loads((DRIVE_V75 / "v7_5_report.json").read_text(encoding="utf-8"))
nm_base, nb_base = v70.verify_t16c_integrity(DRIVE_V75 / "gemma4_31b_v7_5_base.t16c")
nm_sh, nb_sh = v70.verify_t16c_integrity(DRIVE_V75 / "gemma4_31b_v7_5_lod_shared.t16c")
d_ver = [v70.verify_t16c_integrity(DRIVE_V75 / f"gemma4_31b_v7_5_lod_d{d}.t16c") for d in range(12)]
cmeta, clayers = crtr.load_crtr(DRIVE_V75 / "gemma4_31b_v7_5_router.crtr")
print(f"[VERIFIED ALL V7.5 SHARDS] base_mats={nm_base} ({nb_base:,} B) shared_mats={nm_sh} districts=12x{d_ver[0][0]} crtr_layers={len(clayers)}")
print(json.dumps({k: v for k, v in rpt.items() if k != 'layer_cos_summary'}, indent=2))

try:
    from google.colab import runtime
    print("[Auto-Unassign] Disconnecting Colab runtime...")
    runtime.unassign()
except Exception as exc:
    print(f"[Auto-Unassign Note] {exc}")